# Линейная регрессия — sklearn

**Тема:** Линейная регрессия: обучение, метрики качества, train/test split  
**Инструменты:** `sklearn.linear_model.LinearRegression`, `MSE`, `MAE`

---

## Идея

Линейная регрессия ищет прямую $y = kx + b$, которая минимизирует сумму квадратов ошибок между предсказанными и реальными значениями.

**Метрики качества:**
- **MSE** (Mean Squared Error) — среднее квадратичное отклонение, штрафует крупные ошибки сильнее
- **MAE** (Mean Absolute Error) — среднее абсолютное отклонение, более устойчиво к выбросам

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.linear_model import LinearRegression
from sklearn.model_selection import train_test_split
from sklearn.metrics import mean_squared_error, mean_absolute_error

np.random.seed(42)

## 1. Синтетические данные — `y = 5x + 25 + шум`

Генерируем данные с известной формулой, чтобы проверить, насколько точно модель её восстанавливает.

In [ ]:
X = np.linspace(0, 10, 100).reshape(-1, 1)
y = 5 * X + 25 + np.random.randn(100, 1) * 5

X_train, X_test, y_train, y_test = train_test_split(X, y, train_size=0.6, random_state=42)

model = LinearRegression()
model.fit(X_train, y_train)

k = model.coef_[0][0]
b = model.intercept_[0]
print(f"Истинная формула:    y = 5.00x + 25.00")
print(f"Найденные коэффиц.: y = {k:.2f}x + {b:.2f}")

In [ ]:
plt.figure(figsize=(9, 4))
plt.plot(X, 5*X + 25, 'g--', alpha=0.5, label='Истинная формула (y=5x+25)')
plt.plot(X, model.predict(X), 'r-', lw=2, label='Предсказание модели')
plt.scatter(X_train, y_train, alpha=0.6, label='Train')
plt.scatter(X_test, y_test, alpha=0.6, label='Test')
plt.title('Линейная регрессия — синтетические данные')
plt.xlabel('X'); plt.ylabel('y')
plt.legend(); plt.grid(); plt.show()

print(f"Train MSE: {mean_squared_error(y_train, model.predict(X_train)):.2f}")
print(f"Test  MSE: {mean_squared_error(y_test,  model.predict(X_test)):.2f}")
print(f"Train MAE: {mean_absolute_error(y_train, model.predict(X_train)):.2f}")
print(f"Test  MAE: {mean_absolute_error(y_test,  model.predict(X_test)):.2f}")

## 2. Реальные данные — зарплата vs стаж

Датасет: зарплата в зависимости от количества лет опыта.

In [ ]:
url = "https://raw.githubusercontent.com/AnnaShestova/salary-years-simple-linear-regression/master/Salary_Data.csv"
df = pd.read_csv(url)
print(df.head())
print(f"\nРазмер датасета: {df.shape}")

In [ ]:
X = df[['YearsExperience']]
y = df['Salary']

model = LinearRegression()
model.fit(X, y)

k = model.coef_[0]
b = model.intercept_
print(f"Уравнение: Зарплата = {k:.2f} × Стаж + {b:.2f}")
print(f"(+{k:.0f}$ за каждый год опыта)")

plt.figure(figsize=(9, 4))
plt.scatter(X, y, color='blue', label='Реальные данные')
plt.plot(X, model.predict(X), color='red', lw=2, label='Линейная регрессия')
plt.title('Зависимость зарплаты от стажа')
plt.xlabel('Стаж (лет)'); plt.ylabel('Зарплата ($)')
plt.legend(); plt.grid(); plt.show()

print(f"\nПрогноз для 15 лет стажа: ${model.predict([[15]])[0]:,.0f}")

## 3. Реальные данные — расход топлива vs мощность (Auto MPG)

Датасет автомобилей: предсказываем мощность (horsepower) по расходу топлива (mpg).  
Разделение 70/30 — обучение/тест.

In [ ]:
url = "https://raw.githubusercontent.com/plotly/datasets/master/auto-mpg.csv"
df = pd.read_csv(url)
df['horsepower'] = pd.to_numeric(df['horsepower'], errors='coerce')
df = df.dropna(subset=['horsepower'])

X = df[['mpg']].values
y = df['horsepower'].values

X_train, X_test, y_train, y_test = train_test_split(X, y, train_size=0.7, random_state=42)

model = LinearRegression()
model.fit(X_train, y_train)

print(f"Уравнение: horsepower = {model.coef_[0]:.4f} × mpg + {model.intercept_:.4f}")
print("-" * 40)
print(f"Train MSE: {mean_squared_error(y_train, model.predict(X_train)):.2f}")
print(f"Test  MSE: {mean_squared_error(y_test,  model.predict(X_test)):.2f}")
print(f"Train MAE: {mean_absolute_error(y_train, model.predict(X_train)):.2f}")
print(f"Test  MAE: {mean_absolute_error(y_test,  model.predict(X_test)):.2f}")

In [ ]:
x_range = np.linspace(X.min(), X.max(), 100).reshape(-1, 1)

plt.figure(figsize=(9, 4))
plt.scatter(X_train, y_train, alpha=0.5, label='Train')
plt.scatter(X_test, y_test, alpha=0.5, label='Test')
plt.plot(x_range, model.predict(x_range), 'r-', lw=2, label='Модель')
plt.title('Мощность vs Расход топлива')
plt.xlabel('Расход (mpg)'); plt.ylabel('Мощность (horsepower)')
plt.legend(); plt.grid(); plt.show()

## 4. Линейная регрессия через `np.linalg.lstsq` (без sklearn)

Полезно понять, что sklearn делает внутри — тот же МНК, только вручную.

In [ ]:
url = "https://raw.githubusercontent.com/likarajo/petrol_consumption/master/data/petrol_consumption.csv"
df = pd.read_csv(url)

X = df[['Petrol_Consumption']].values
y = df['Population_Driver_licence(%)'].values

# Добавляем столбец единиц для свободного члена b
X_matrix = np.c_[X, np.ones(X.shape[0])]
coeffs = np.linalg.lstsq(X_matrix, y, rcond=None)[0]

print(f"Наклон (k):        {coeffs[0]:.6f}")
print(f"Свободный член (b):{coeffs[1]:.4f}")
print(f"Уравнение: y = {coeffs[0]:.4f} × x + {coeffs[1]:.4f}")

plt.figure(figsize=(9, 4))
plt.scatter(X, y, color='blue', label='Данные')
plt.plot(X, coeffs[0]*X + coeffs[1], 'r-', lw=2, label='МНК (вручную)')
plt.xlabel('Потребление бензина')
plt.ylabel('Доля водителей (%)')
plt.legend(); plt.grid(); plt.show()

## Итог

| Подход | Инструмент | Особенность |
|---|---|---|
| sklearn | `LinearRegression` | Удобный API, train/test split, метрики |
| NumPy | `np.linalg.lstsq` | Прозрачно — видна математика |

**Train MSE ≈ Test MSE** → модель обобщается нормально  
**Train MSE << Test MSE** → переобучение — модель запомнила обучающие данные